# 10 - Word Similarity in NLP

This notebook explores semantic similarity using WordNet and NLTK.

## Learning Objectives
- Understand the difference between word similarity and synset similarity.
- Select WordNet synsets with appropriate meanings and parts of speech.
- Calculate Wu-Palmer, path-based, and Leacock-Chodorow similarity.
- Handle missing similarity scores safely.
- Find common hypernyms and compare multiple word pairs.

> Similarity scores depend on the selected WordNet senses. They are not general-purpose, context-aware embeddings.

## 1. Import WordNet

WordNet is distributed separately from the NLTK package. If the corpus is missing, use the download cell below.

In [ ]:
# Import WordNet from NLTK
from nltk.corpus import wordnet as wn


### Download WordNet data if needed

In [ ]:
# Run these downloads only if NLTK reports that the resources are missing
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")


## 2. Understand Synsets Before Comparing

A word can have multiple meanings, and each meaning may have its own synset.
For a fair comparison, choose synsets that match the intended meaning and use compatible parts of speech.

In the examples below, we use noun synsets for `cat`, `dog`, `wolf`, and `car`.

In [ ]:
# Select the first noun synset for each sample word
cat_synsets = wn.synsets("cat", pos=wn.NOUN)
dog_synsets = wn.synsets("dog", pos=wn.NOUN)
wolf_synsets = wn.synsets("wolf", pos=wn.NOUN)
car_synsets = wn.synsets("car", pos=wn.NOUN)

# Store a synset only when WordNet returned at least one result
cat = cat_synsets[0] if cat_synsets else None
dog = dog_synsets[0] if dog_synsets else None
wolf = wolf_synsets[0] if wolf_synsets else None
car = car_synsets[0] if car_synsets else None

# Display selected synsets and their definitions
for word, synset in [("cat", cat), ("dog", dog), ("wolf", wolf), ("car", car)]:
    if synset is not None:
        print(f"{word}: {synset.name()} - {synset.definition()}")
    else:
        print(f"No noun synset found for {word!r}.")


## 3. Path Similarity

`path_similarity()` compares two synsets using the shortest path between them in WordNet's hierarchy.

- Scores are generally between `0` and `1`.
- A larger score indicates a closer connection in the hierarchy.
- The result can be `None` when a usable path is unavailable.

This is a structural measure, not a measure of contextual meaning in a sentence.

In [ ]:
# Calculate path similarity safely
if cat is not None and dog is not None:
    path_score = cat.path_similarity(dog)
    print("Cat-Dog path similarity:", path_score)
else:
    print("Cannot compare cat and dog because a synset is missing.")


## 4. Wu-Palmer Similarity

`wup_similarity()` (Wu-Palmer similarity) uses the depth of the two synsets
and their lowest common subsumer in the taxonomy.

It is useful for comparing related concepts in WordNet, but results still depend
on the selected senses.

In [ ]:
# Calculate Wu-Palmer similarity for selected noun synsets
if cat is not None and dog is not None:
    wup_score = cat.wup_similarity(dog)
    print("Cat-Dog Wu-Palmer similarity:", wup_score)
else:
    print("Cannot calculate Wu-Palmer similarity: a synset is missing.")


## 5. Leacock-Chodorow Similarity

`lch_similarity()` uses the shortest path between synsets and the taxonomy's depth.
It is defined for compatible WordNet synsets, typically of the same part of speech.

Unlike path similarity and Wu-Palmer similarity, its score is not restricted to the interval from 0 to 1.

In [ ]:
# Calculate Leacock-Chodorow similarity for two noun synsets
if cat is not None and dog is not None:
    try:
        lch_score = cat.lch_similarity(dog)
        print("Cat-Dog Leacock-Chodorow similarity:", lch_score)
    except (ValueError, RuntimeError) as error:
        print("Could not calculate LCH similarity:", error)
else:
    print("Cannot calculate LCH similarity: a synset is missing.")


## 6. Compare Several Word Pairs

The helper function below checks whether both words have a noun synset before calculating similarity.
For this demonstration, it uses the first noun synset returned by WordNet. In a real application, inspect the definitions and select senses that match the intended meanings.

In [ ]:
def compare_noun_similarity(word1, word2):
    """Print WordNet similarity scores for the first noun synsets of two words."""
    synsets1 = wn.synsets(word1, pos=wn.NOUN)
    synsets2 = wn.synsets(word2, pos=wn.NOUN)

    # Check that both words have at least one noun synset
    if not synsets1 or not synsets2:
        print(f"Could not find noun synsets for {word1!r} or {word2!r}.")
        return

    synset1 = synsets1[0]
    synset2 = synsets2[0]

    # Display the meanings used in this comparison
    print(f"Word 1: {synset1.name()} - {synset1.definition()}")
    print(f"Word 2: {synset2.name()} - {synset2.definition()}")

    # Calculate each measure
    print("Path similarity:", synset1.path_similarity(synset2))
    print("Wu-Palmer similarity:", synset1.wup_similarity(synset2))

    # LCH similarity may raise an error for incompatible synsets
    try:
        print("Leacock-Chodorow similarity:", synset1.lch_similarity(synset2))
    except (ValueError, RuntimeError) as error:
        print("Leacock-Chodorow similarity unavailable:", error)

    print("-" * 60)


# Compare related and less-related concepts
compare_noun_similarity("cat", "dog")
compare_noun_similarity("cat", "wolf")
compare_noun_similarity("cat", "car")


## 7. Lowest Common Hypernyms

The lowest common hypernym is the most specific shared ancestor returned by WordNet for two synsets.
It can help explain why two concepts are related.

In [ ]:
# Find the lowest common hypernyms for cat and dog
if cat is not None and dog is not None:
    common_hypernyms = cat.lowest_common_hypernyms(dog)

    for synset in common_hypernyms:
        print(synset.name(), ":", synset.definition())
else:
    print("Cannot find common hypernyms because a synset is missing.")


## 8. Compare Multiple Metrics in a Table

Different metrics have different scales and definitions. Do not compare raw values across metrics as if they were directly interchangeable.

In [ ]:
# Build a small comparison table using Python's standard library
pairs = [("cat", "dog"), ("cat", "wolf"), ("cat", "car")]
results = []

for word1, word2 in pairs:
    synsets1 = wn.synsets(word1, pos=wn.NOUN)
    synsets2 = wn.synsets(word2, pos=wn.NOUN)

    if not synsets1 or not synsets2:
        results.append((word1, word2, None, None, None))
        continue

    synset1, synset2 = synsets1[0], synsets2[0]

    path_score = synset1.path_similarity(synset2)
    wup_score = synset1.wup_similarity(synset2)

    try:
        lch_score = synset1.lch_similarity(synset2)
    except (ValueError, RuntimeError):
        lch_score = None

    results.append((word1, word2, path_score, wup_score, lch_score))

# Print the results in aligned columns
print(f"{'Word 1':<10} {'Word 2':<10} {'Path':>10} {'Wu-Palmer':>12} {'LCH':>10}")
print("-" * 56)

for word1, word2, path_score, wup_score, lch_score in results:
    print(
        f"{word1:<10} {word2:<10} "
        f"{str(path_score):>10} {str(wup_score):>12} {str(lch_score):>10}"
    )


## 9. Exercises

### Exercise 1: Try Other Word Pairs
Use `compare_noun_similarity()` to compare:
- `horse` and `donkey`
- `apple` and `orange`
- `car` and `bicycle`

### Exercise 2: Inspect the Senses
Find all noun synsets for `bank`. Choose two different senses and explain why the selected meaning matters when calculating similarity.

### Exercise 3: Common Hypernyms
Find the lowest common hypernym of `wolf` and `dog`.

### Exercise 4: Interpret the Scores
Why should we not conclude that a larger raw LCH score is directly comparable to a larger path similarity score?

### Exercise starter code

In [ ]:
# Exercise: compare horse and donkey
compare_noun_similarity("horse", "donkey")

# TODO: inspect the noun synsets for "bank"
bank_noun_synsets = wn.synsets("bank", pos=wn.NOUN)

for synset in bank_noun_synsets:
    print(synset.name(), ":", synset.definition())


## 10. Summary

- WordNet similarity compares selected synsets, not ambiguous words in isolation.
- `path_similarity()` uses the shortest path in the taxonomy.
- `wup_similarity()` uses taxonomy depth and a shared ancestor.
- `lch_similarity()` uses path length and taxonomy depth and has a different score scale.
- Always inspect synset definitions and handle `None` results or incompatible comparisons.
- These measures are based on WordNet's lexical hierarchy; they are not embedding-based contextual similarity.

**Next step:** apply these concepts to more advanced lexical or semantic NLP tasks.